In [95]:
import numpy as np
import matplotlib as matplotlib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from gensim.models import Word2Vec

In [63]:
import re

def clean_text(text: str) -> str:
    if text == None:
        return ''
    
    text = re.sub(r"[^a-zA-Zа-яА-Я0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def lower_text(text: str) -> str:
    return text.lower()

def transform_text(text: str) -> list:
    clean = clean_text(text)
    clean = lower_text(clean)
    return clean.split()

In [3]:
data = pd.read_csv("train.csv")
data

,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."
...,...,...
48660,5,"Удобный, но маленький и ещё не обновили как др..."
48661,2,"Постоянно обман в цене,написанна сумма на акци..."
48662,2,Очень хочется пожелать этому магазину стать та...
48663,5,"Нравится ваш магазин, персонал одекватный, пор..."


In [4]:
X, y = data.drop(columns = ['rate']), pd.DataFrame(data['rate'])
X

,text
0,Очень понравилось. Были в начале марта с соба...
1,В целом магазин устраивает.\nАссортимент позво...
2,"Очень хорошо что открылась 5 ка, теперь не над..."
3,Пятёрочка громко объявила о том как она заботи...
4,"Тесно, вечная сутолока, между рядами трудно ра..."
...,...
48660,"Удобный, но маленький и ещё не обновили как др..."
48661,"Постоянно обман в цене,написанна сумма на акци..."
48662,Очень хочется пожелать этому магазину стать та...
48663,"Нравится ваш магазин, персонал одекватный, пор..."


In [5]:
y

,rate
0,4
1,5
2,5
3,3
4,3
...,...
48660,5
48661,2
48662,2
48663,5


In [64]:
X_transformed = X.copy()
X_transformed['tokens'] = X_transformed['text'].astype(str).apply(transform_text)
X_transformed

,text,tokens
0,Очень понравилось. Были в начале марта с соба...,"[очень, понравилось, были, в, начале, марта, с..."
1,В целом магазин устраивает.\nАссортимент позво...,"[в, целом, магазин, устраивает, ассортимент, п..."
2,"Очень хорошо что открылась 5 ка, теперь не над...","[очень, хорошо, что, открылась, 5, ка, теперь,..."
3,Пятёрочка громко объявила о том как она заботи...,"[пятрочка, громко, объявила, о, том, как, она,..."
4,"Тесно, вечная сутолока, между рядами трудно ра...","[тесно, вечная, сутолока, между, рядами, трудн..."
...,...,...
48660,"Удобный, но маленький и ещё не обновили как др...","[удобный, но, маленький, и, ещ, не, обновили, ..."
48661,"Постоянно обман в цене,написанна сумма на акци...","[постоянно, обман, в, цененаписанна, сумма, на..."
48662,Очень хочется пожелать этому магазину стать та...,"[очень, хочется, пожелать, этому, магазину, ст..."
48663,"Нравится ваш магазин, персонал одекватный, пор...","[нравится, ваш, магазин, персонал, одекватный,..."


In [96]:
sentences = X_transformed['tokens'].tolist()
fasttext_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

In [97]:
def get_w2v_vector(text_tokens, model):
    vectors = [model.wv[word] for word in text_tokens if word in model.wv]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

In [130]:
X_transformed['fasttext_vector'] = X_transformed['tokens'].apply(lambda x: get_w2v_vector(x, fasttext_model))
X_transformed

,text,tokens,fasttext_vector
0,Очень понравилось. Были в начале марта с соба...,"[очень, понравилось, были, в, начале, марта, с...","[-0.49751934, 0.34158278, 0.076608986, 0.14842..."
1,В целом магазин устраивает.\nАссортимент позво...,"[в, целом, магазин, устраивает, ассортимент, п...","[-0.758033, 0.23333447, -0.18440871, 0.2132701..."
2,"Очень хорошо что открылась 5 ка, теперь не над...","[очень, хорошо, что, открылась, 5, ка, теперь,...","[0.0023798575, 0.4078912, 0.007815858, 0.43086..."
3,Пятёрочка громко объявила о том как она заботи...,"[пятрочка, громко, объявила, о, том, как, она,...","[-0.3754474, 0.5510315, -0.019410059, 0.545977..."
4,"Тесно, вечная сутолока, между рядами трудно ра...","[тесно, вечная, сутолока, между, рядами, трудн...","[-0.5103968, 0.5696486, -0.29634377, 0.0150827..."
...,...,...,...
48660,"Удобный, но маленький и ещё не обновили как др...","[удобный, но, маленький, и, ещ, не, обновили, ...","[-0.2979234, 0.5824968, -0.21163408, 0.4627335..."
48661,"Постоянно обман в цене,написанна сумма на акци...","[постоянно, обман, в, цененаписанна, сумма, на...","[-0.6791262, 0.28558773, -0.06963397, 0.277582..."
48662,Очень хочется пожелать этому магазину стать та...,"[очень, хочется, пожелать, этому, магазину, ст...","[-0.36481765, 0.44278717, -0.022034712, 0.3238..."
48663,"Нравится ваш магазин, персонал одекватный, пор...","[нравится, ваш, магазин, персонал, одекватный,...","[-0.36742976, -0.19902708, 0.59708494, -0.2959..."


In [135]:
X_vectors = np.vstack(X_transformed['fasttext_vector'].values)
X_transformed['fasttext_vector'] = np.vstack(X_transformed['fasttext_vector'].values)
X_transformed

,text,tokens,fasttext_vector
0,Очень понравилось. Были в начале марта с соба...,"[очень, понравилось, были, в, начале, марта, с...",-0.497519
1,В целом магазин устраивает.\nАссортимент позво...,"[в, целом, магазин, устраивает, ассортимент, п...",-0.758033
2,"Очень хорошо что открылась 5 ка, теперь не над...","[очень, хорошо, что, открылась, 5, ка, теперь,...",0.002380
3,Пятёрочка громко объявила о том как она заботи...,"[пятрочка, громко, объявила, о, том, как, она,...",-0.375447
4,"Тесно, вечная сутолока, между рядами трудно ра...","[тесно, вечная, сутолока, между, рядами, трудн...",-0.510397
...,...,...,...
48660,"Удобный, но маленький и ещё не обновили как др...","[удобный, но, маленький, и, ещ, не, обновили, ...",-0.297923
48661,"Постоянно обман в цене,написанна сумма на акци...","[постоянно, обман, в, цененаписанна, сумма, на...",-0.679126
48662,Очень хочется пожелать этому магазину стать та...,"[очень, хочется, пожелать, этому, магазину, ст...",-0.364818
48663,"Нравится ваш магазин, персонал одекватный, пор...","[нравится, ваш, магазин, персонал, одекватный,...",-0.367430


In [136]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed.drop(columns=['tokens']), y, random_state=42, test_size=0.15)

In [137]:
model = CatBoostClassifier(
    iterations=250,
    random_seed=42,
    text_features=['text'],
    loss_function="MultiClass",
    eval_metric="TotalF1"
)

model.fit(
    X_train,
    y_train,
    verbose=True
)

Learning rate set to 0.308683
0:	learn: 0.5960412	total: 3.08s	remaining: 12m 47s
1:	learn: 0.5841232	total: 5.55s	remaining: 11m 28s
2:	learn: 0.5939159	total: 7.04s	remaining: 9m 40s
3:	learn: 0.5963946	total: 8.19s	remaining: 8m 23s
4:	learn: 0.5967198	total: 9.58s	remaining: 7m 49s
5:	learn: 0.5946407	total: 10.8s	remaining: 7m 18s
6:	learn: 0.5943503	total: 12s	remaining: 6m 57s
7:	learn: 0.5967885	total: 13.2s	remaining: 6m 39s
8:	learn: 0.5985211	total: 14.4s	remaining: 6m 26s
9:	learn: 0.5985782	total: 15.6s	remaining: 6m 13s
10:	learn: 0.5997158	total: 17s	remaining: 6m 8s
11:	learn: 0.6002517	total: 18.1s	remaining: 5m 58s
12:	learn: 0.6007410	total: 19.3s	remaining: 5m 51s
13:	learn: 0.6016970	total: 20.5s	remaining: 5m 45s


KeyboardInterrupt: 

In [119]:
prediction = model.predict(X_test)

In [120]:
prediction

array([[5],
       [5],
       [5],
       ...,
       [4],
       [4],
       [5]], dtype=int64)

In [121]:
f1_score(y_test, prediction, average='weighted')

c:\Users\Артём\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\Артём\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\Артём\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\Артём\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning:

0.5703866965005356

# Предсказания

In [122]:
data_test = pd.read_csv('test.csv')
data_test

,index,text
0,0,Очень хороший магазин и сотрудники приятный
1,1,"Самый обычный продуктовый магазин. Есть сыры,..."
2,2,Всё хорошо и комфортно
3,3,"Маленький филиальчик, все необходимое есть. Дв..."
4,4,Плохо относятся к клиентам!!!!!\n
...,...,...
12162,12162,Персонал вежливый . Большой ассортимент
12163,12163,Скидки на сыры. Скидки на алкоголь. Приемлимые...
12164,12164,"Рядом с домом, неплохая пятерочка, персонал хо..."
12165,12165,Хороший магазин у дома. Кассиры приветливые. П...


In [123]:
X_test_for_prediction = data_test.copy()
X_test_for_prediction['tokens'] = X_test_for_prediction['text'].astype(str).apply(transform_text)
X_test_for_prediction

,index,text,tokens
0,0,Очень хороший магазин и сотрудники приятный,"[очень, хороший, магазин, и, сотрудники, прият..."
1,1,"Самый обычный продуктовый магазин. Есть сыры,...","[самый, обычный, продуктовый, магазин, есть, с..."
2,2,Всё хорошо и комфортно,"[вс, хорошо, и, комфортно]"
3,3,"Маленький филиальчик, все необходимое есть. Дв...","[маленький, филиальчик, все, необходимое, есть..."
4,4,Плохо относятся к клиентам!!!!!\n,"[плохо, относятся, к, клиентам]"
...,...,...,...
12162,12162,Персонал вежливый . Большой ассортимент,"[персонал, вежливый, большой, ассортимент]"
12163,12163,Скидки на сыры. Скидки на алкоголь. Приемлимые...,"[скидки, на, сыры, скидки, на, алкоголь, прием..."
12164,12164,"Рядом с домом, неплохая пятерочка, персонал хо...","[рядом, с, домом, неплохая, пятерочка, персона..."
12165,12165,Хороший магазин у дома. Кассиры приветливые. П...,"[хороший, магазин, у, дома, кассиры, приветлив..."


In [124]:
sentences_fasttext = X_test_for_prediction['tokens'].tolist()

In [129]:
X_test_for_prediction['fasttext_vector'] = X_test_for_prediction['tokens'].apply(lambda x: get_w2v_vector(x, fasttext_model))
X_test_for_prediction

,index,text,tokens,fasttext_vector
0,0,Очень хороший магазин и сотрудники приятный,"[очень, хороший, магазин, и, сотрудники, прият...","[-0.07407827, 0.018500691, 0.26255846, 0.36645..."
1,1,"Самый обычный продуктовый магазин. Есть сыры,...","[самый, обычный, продуктовый, магазин, есть, с...","[-0.44049877, 0.3419752, 0.2468588, 0.37458986..."
2,2,Всё хорошо и комфортно,"[вс, хорошо, и, комфортно]","[-0.026332337, 0.38226357, 0.26526514, 0.32936..."
3,3,"Маленький филиальчик, все необходимое есть. Дв...","[маленький, филиальчик, все, необходимое, есть...","[-0.41264296, 0.21401434, -0.25634763, 0.38161..."
4,4,Плохо относятся к клиентам!!!!!\n,"[плохо, относятся, к, клиентам]","[-0.7670591, 0.18310079, 0.039925203, 1.089613..."
...,...,...,...,...
12162,12162,Персонал вежливый . Большой ассортимент,"[персонал, вежливый, большой, ассортимент]","[-0.119821995, -0.84252554, -0.01529488, 0.795..."
12163,12163,Скидки на сыры. Скидки на алкоголь. Приемлимые...,"[скидки, на, сыры, скидки, на, алкоголь, прием...","[-0.9388268, 0.10878629, -0.23902227, 0.431342..."
12164,12164,"Рядом с домом, неплохая пятерочка, персонал хо...","[рядом, с, домом, неплохая, пятерочка, персона...","[-0.03698292, 0.34072164, 0.17645906, 0.260619..."
12165,12165,Хороший магазин у дома. Кассиры приветливые. П...,"[хороший, магазин, у, дома, кассиры, приветлив...","[-0.8776378, 0.2023874, 0.18634656, 0.11868016..."


In [126]:
X_vectors_test = np.vstack(X_test_for_prediction['fasttext_vector'].values)

In [127]:
pred = pd.DataFrame(model.predict(X_vectors_test), columns=['rate'])
pred.reset_index(inplace=True)
pred

,index,rate
0,0,5
1,1,5
2,2,5
3,3,5
4,4,1
...,...,...
12162,12162,5
12163,12163,5
12164,12164,5
12165,12165,5


In [128]:

pred.to_csv("submission.csv", index=False)